# Chapter 3.7: Model Loader & Weight Loading

## Learning Objectives

By the end of this notebook, you will:

1. Understand the model loading pipeline from disk to GPU
2. Know the Safetensors format and why it's preferred
3. Understand weight sharding for tensor parallelism
4. Know how quantized weights are loaded (GPTQ, AWQ)
5. Understand the ModelRegistry and how models are registered
6. Know weight mapping and transformation techniques
7. Understand memory-efficient loading strategies

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part3/chapter_3.7_model_loader.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part3/chapter_3.7_model_loader.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Model Loading Pipeline Overview

```
HuggingFace Hub / Local Path
       │
       ▼
┌──────────────────┐
│ Download Weights  │  (huggingface_hub / cached)
│ (.safetensors)    │
└────────┬─────────┘
         │
         ▼
┌──────────────────┐
│ ModelRegistry     │  (find the right model class)
│ "llama" → LlamaForCausalLM
└────────┬─────────┘
         │
         ▼
┌──────────────────┐
│ Create Model     │  (empty weights on meta device)
│ Architecture     │
└────────┬─────────┘
         │
         ▼
┌──────────────────┐
│ Load Weights     │  (iterate through weight files)
│ - Shard for TP   │  (split across GPUs)
│ - Quantize       │  (if quantized model)
│ - Map names      │  (HF names → vLLM names)
└────────┬─────────┘
         │
         ▼
┌──────────────────┐
│ Model on GPU     │  (ready for inference)
└──────────────────┘
```

In [ ]:
# Source: vllm/model_executor/model_loader/loader.py (simplified)

loader_code = '''
def get_model(
    model_config: ModelConfig,
    device_config: DeviceConfig,
    ...
) -> nn.Module:
    """Load a model from configuration."""
    
    # Step 1: Determine which loader to use
    loader = _get_model_loader(model_config)
    # Options:
    #   DefaultModelLoader: standard weight loading
    #   DummyModelLoader: for testing (random weights)
    #   TensorizeLoader: for pre-serialized weights
    #   ShardedStateLoader: for pre-sharded weights
    
    # Step 2: Load the model
    return loader.load_model(
        model_config=model_config,
        device_config=device_config,
        ...
    )


class DefaultModelLoader:
    """Standard model loader for HuggingFace models."""
    
    def load_model(self, model_config, device_config, ...):
        # Step 1: Get model architecture class from registry
        model_class = ModelRegistry.resolve_model_cls(
            model_config.architectures
        )
        # e.g., "LlamaForCausalLM" → vllm.model_executor.models.llama.LlamaForCausalLM
        
        # Step 2: Create model with empty weights
        with set_default_torch_dtype(model_config.dtype):
            with torch.device("meta"):
                # "meta" device: creates tensors without allocating memory
                model = model_class(
                    config=model_config.hf_config,
                    cache_config=cache_config,
                    ...
                )
        
        # Step 3: Load weights from disk
        model.load_weights(
            self._get_weights_iterator(model_config)
        )
        
        # Step 4: Move to target device
        model = model.to(device_config.device)
        
        return model
'''
print(loader_code)

## 2. Safetensors Format

In [ ]:
safetensors_info = """
Safetensors Format — Why vLLM Prefers It
═════════════════════════════════════════

What is Safetensors?
  A file format for storing tensors, created by HuggingFace.
  Alternative to PyTorch's .bin files (pickle-based).

File Structure:
  ┌──────────────┐
  │ Header (JSON) │  Metadata: tensor names, shapes, dtypes, offsets
  ├──────────────┤
  │ Tensor Data   │  Raw binary data (no serialization overhead)
  │ (contiguous)  │
  └──────────────┘

Advantages over .bin (pickle):
  1. SECURITY: No arbitrary code execution (pickle can run code)
  2. SPEED: Memory-mapped loading (mmap) — no copy needed
  3. LAZY LOADING: Can load individual tensors without reading entire file
  4. ZERO-COPY: Tensors are stored contiguously, direct GPU transfer

Loading Flow:
  1. Open .safetensors file (memory-mapped)
  2. Read header → get tensor names, shapes, byte offsets
  3. For each needed tensor:
     a. Compute byte range from header
     b. Read tensor data directly into GPU memory
     c. Reshape to correct dimensions

Performance:
  LLaMA-7B loading time:
    .bin (pickle): ~30-60 seconds
    .safetensors:  ~10-20 seconds (2-3x faster)

vLLM Weight File Discovery:
  1. Look for *.safetensors files first
  2. Fall back to *.bin files
  3. Use model.safetensors.index.json for sharded models
"""
print(safetensors_info)

In [ ]:
# Weight iterator — how weights are loaded one by one

weight_iterator_code = '''
def _get_weights_iterator(self, model_config):
    """Create an iterator that yields (name, tensor) pairs."""
    
    # Find all weight files
    model_path = model_config.model
    
    # Check for safetensors
    safetensors_files = glob.glob(
        os.path.join(model_path, "*.safetensors")
    )
    
    if safetensors_files:
        # Use safetensors loading
        for weight_file in safetensors_files:
            with safe_open(weight_file, framework="pt") as f:
                for name in f.keys():
                    tensor = f.get_tensor(name)
                    yield name, tensor
    else:
        # Fall back to pickle (.bin files)
        bin_files = glob.glob(
            os.path.join(model_path, "*.bin")
        )
        for weight_file in bin_files:
            state_dict = torch.load(weight_file, map_location="cpu")
            for name, tensor in state_dict.items():
                yield name, tensor
            del state_dict
            torch.cuda.empty_cache()
'''
print(weight_iterator_code)

## 3. ModelRegistry — How Models Are Registered

In [ ]:
registry_code = '''
# Source: vllm/model_executor/models/registry.py

# The registry maps HuggingFace model architecture names
# to vLLM model implementation classes.

_MODELS = {
    # LLaMA family
    "LlamaForCausalLM": ("llama", "LlamaForCausalLM"),
    "MistralForCausalLM": ("llama", "LlamaForCausalLM"),  # Mistral uses LLaMA arch
    
    # GPT-2
    "GPT2LMHeadModel": ("gpt2", "GPT2LMHeadModel"),
    
    # Qwen
    "Qwen2ForCausalLM": ("qwen2", "Qwen2ForCausalLM"),
    
    # Phi
    "PhiForCausalLM": ("phi", "PhiForCausalLM"),
    
    # Gemma
    "GemmaForCausalLM": ("gemma", "GemmaForCausalLM"),
    
    # ... 50+ more models
}


class ModelRegistry:
    @staticmethod
    def resolve_model_cls(
        architectures: List[str],
    ) -> Type[nn.Module]:
        """Resolve a model class from HF architecture name."""
        for arch in architectures:
            if arch in _MODELS:
                module_name, class_name = _MODELS[arch]
                module = importlib.import_module(
                    f"vllm.model_executor.models.{module_name}"
                )
                return getattr(module, class_name)
        
        raise ValueError(
            f"Model architectures {architectures} are not supported."
        )
    
    @staticmethod
    def is_supported(architectures: List[str]) -> bool:
        return any(arch in _MODELS for arch in architectures)


# How HF model config maps to vLLM:
# 1. Load config.json from model directory
# 2. Read "architectures" field
#    e.g., {"architectures": ["LlamaForCausalLM"]}
# 3. Look up in _MODELS registry
# 4. Import and instantiate the vLLM model class
'''
print(registry_code)

## 4. Weight Sharding for Tensor Parallelism

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Column parallel
ax = axes[0]
ax.set_title('Column-Parallel Linear\n(QKV Projection)', fontsize=12, fontweight='bold')
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')

# Original weight matrix
rect = mpatches.FancyBboxPatch((0.5, 3), 3, 2.5, boxstyle="round,pad=0.1",
                                facecolor='lightyellow', edgecolor='black')
ax.add_patch(rect)
ax.text(2, 5, 'W [hidden, 4*hidden]', ha='center', fontsize=8)

# Split columns
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
for i in range(4):
    rect = mpatches.FancyBboxPatch((5.5 + i*1.1, 3.5+i*0.4), 0.9, 1.8-i*0.1,
                                    boxstyle="round,pad=0.05",
                                    facecolor=colors[i], edgecolor='black', alpha=0.7)
    ax.add_patch(rect)
    ax.text(5.95 + i*1.1, 2.8, f'GPU {i}', ha='center', fontsize=7)

ax.annotate('', xy=(5.3, 4.2), xytext=(3.7, 4.2),
            arrowprops=dict(arrowstyle='->', lw=2))
ax.text(4.5, 4.6, 'Split\ncolumns', ha='center', fontsize=8)
ax.text(5, 1.5, 'Each GPU: W[:, i*cols:(i+1)*cols]\nNo communication needed', 
        ha='center', fontsize=8, style='italic')

# Row parallel
ax = axes[1]
ax.set_title('Row-Parallel Linear\n(Output Projection)', fontsize=12, fontweight='bold')
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')

rect = mpatches.FancyBboxPatch((0.5, 3), 3, 2.5, boxstyle="round,pad=0.1",
                                facecolor='lightyellow', edgecolor='black')
ax.add_patch(rect)
ax.text(2, 5, 'W [4*hidden, hidden]', ha='center', fontsize=8)

for i in range(4):
    rect = mpatches.FancyBboxPatch((5.5, 3 + i*0.6), 3, 0.5,
                                    boxstyle="round,pad=0.05",
                                    facecolor=colors[i], edgecolor='black', alpha=0.7)
    ax.add_patch(rect)
    ax.text(9, 3.25 + i*0.6, f'GPU {i}', fontsize=7)

ax.annotate('', xy=(5.3, 4.2), xytext=(3.7, 4.2),
            arrowprops=dict(arrowstyle='->', lw=2))
ax.text(4.5, 4.6, 'Split\nrows', ha='center', fontsize=8)
ax.text(5, 1.5, 'Each GPU: W[i*rows:(i+1)*rows, :]\nAllReduce after matmul', 
        ha='center', fontsize=8, style='italic')

plt.tight_layout()
plt.show()

In [ ]:
# Weight sharding implementation

sharding_code = '''
# In vLLM model implementations (e.g., llama.py)

class LlamaAttention(nn.Module):
    def __init__(self, config, ...):
        # QKV projection: column-parallel
        # Original: [hidden_size, num_heads * head_dim * 3]
        # Sharded: each GPU gets [hidden_size, num_heads/TP * head_dim * 3]
        self.qkv_proj = QKVParallelLinear(
            hidden_size=config.hidden_size,
            head_size=self.head_dim,
            total_num_heads=config.num_attention_heads,
            total_num_kv_heads=config.num_key_value_heads,
        )
        
        # Output projection: row-parallel  
        # Original: [num_heads * head_dim, hidden_size]
        # Sharded: each GPU gets [num_heads/TP * head_dim, hidden_size]
        self.o_proj = RowParallelLinear(
            input_size=config.num_attention_heads * self.head_dim,
            output_size=config.hidden_size,
        )


class QKVParallelLinear(ColumnParallelLinear):
    """Column-parallel linear for QKV projections.
    
    Special handling because Q has more heads than K,V in GQA.
    """
    
    def weight_loader(self, param, loaded_weight, loaded_shard_id):
        """Load a shard of the QKV weight.
        
        loaded_shard_id: "q", "k", or "v"
        """
        tp_rank = get_tensor_model_parallel_rank()
        tp_size = get_tensor_model_parallel_world_size()
        
        if loaded_shard_id == "q":
            # Q: split across TP for num_attention_heads
            shard_size = self.num_heads // tp_size * self.head_dim
            start = tp_rank * shard_size
            loaded_weight = loaded_weight[start:start + shard_size]
        elif loaded_shard_id in ("k", "v"):
            # K,V: split across TP for num_kv_heads
            shard_size = self.num_kv_heads // tp_size * self.head_dim
            start = tp_rank * shard_size
            loaded_weight = loaded_weight[start:start + shard_size]
        
        # Copy into the parameter at the correct offset
        param_data = param.data
        # ... copy loaded_weight into the right position
'''
print(sharding_code)

## 5. Quantized Weight Loading

In [ ]:
quant_loading = """
Quantized Weight Loading
════════════════════════

GPTQ (GPT Quantization):
  - Weights stored as int4/int8 + scale + zero_point
  - Each weight file contains:
    - qweight: packed int4 weights
    - qzeros: zero points
    - scales: dequantization scales
    - g_idx: group indices (for group quantization)
  
  Loading:
    1. Read packed int4 tensors from safetensors
    2. Repack for the target CUDA kernel format
    3. Shard scales and zeros across TP ranks
    4. Store in the model's quantized linear layers

AWQ (Activation-Aware Quantization):
  - Similar to GPTQ but with different quantization scheme
  - Weights: qweight (int4), qzeros, scales
  - Different packing format than GPTQ
  
  Loading:
    1. Read AWQ-format weights
    2. Convert to vLLM's internal format
    3. Shard for TP if needed

Memory Comparison (LLaMA-7B):
  FP16:  ~14 GB
  INT8:  ~7 GB  (2x compression)
  INT4:  ~3.5 GB (4x compression)
  
  This means 4x more KV cache space available!

Quantization Method Selection:
  # In config.json or EngineArgs:
  quantization = "gptq"   # or "awq", "squeezellm", "marlin"
  
  # vLLM selects the right quantized linear layer:
  if quantization == "gptq":
      linear_cls = GPTQLinear
  elif quantization == "awq":
      linear_cls = AWQLinear
  # etc.
"""
print(quant_loading)

In [ ]:
# Memory comparison visualization

import matplotlib.pyplot as plt

models = ['LLaMA-7B', 'LLaMA-13B', 'LLaMA-70B']
fp16_sizes = [14, 26, 140]  # GB
int8_sizes = [7, 13, 70]
int4_sizes = [3.5, 6.5, 35]

x = np.arange(len(models))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width, fp16_sizes, width, label='FP16', color='#FF6B6B')
bars2 = ax.bar(x, int8_sizes, width, label='INT8', color='#4ECDC4')
bars3 = ax.bar(x + width, int4_sizes, width, label='INT4 (GPTQ/AWQ)', color='#45B7D1')

ax.set_ylabel('Model Size (GB)')
ax.set_title('Model Size by Quantization', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()

# A100 80GB line
ax.axhline(y=80, color='red', linestyle='--', alpha=0.5, label='A100 80GB')
ax.text(2.3, 82, 'A100 80GB', fontsize=9, color='red')

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 1,
                f'{h}GB', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

print("Key insight: INT4 quantization lets you fit LLaMA-70B on a single A100!")
print("Without quantization, you need at least 2x A100s for FP16.")

## 6. Weight Name Mapping

In [ ]:
weight_mapping = '''
Weight Name Mapping: HuggingFace → vLLM
═════════════════════════════════════════

HuggingFace model weights use specific naming conventions.
vLLM model implementations may use different names.
The load_weights() method handles the mapping.

Example: LLaMA model weight names

HuggingFace Name                           vLLM Internal
────────────────────────────────────────    ────────────────────────
model.embed_tokens.weight                  model.embed_tokens.weight
model.layers.0.self_attn.q_proj.weight     model.layers.0.self_attn.qkv_proj.weight (merged)
model.layers.0.self_attn.k_proj.weight     model.layers.0.self_attn.qkv_proj.weight (merged)
model.layers.0.self_attn.v_proj.weight     model.layers.0.self_attn.qkv_proj.weight (merged)
model.layers.0.self_attn.o_proj.weight     model.layers.0.self_attn.o_proj.weight
model.layers.0.mlp.gate_proj.weight        model.layers.0.mlp.gate_up_proj.weight (merged)
model.layers.0.mlp.up_proj.weight          model.layers.0.mlp.gate_up_proj.weight (merged)
model.layers.0.mlp.down_proj.weight        model.layers.0.mlp.down_proj.weight
model.layers.0.input_layernorm.weight      model.layers.0.input_layernorm.weight
model.layers.0.post_attention_layernorm.weight  (same)
model.norm.weight                          model.norm.weight
lm_head.weight                             lm_head.weight

Key Optimization: Merged QKV and Gate+Up projections
  vLLM merges Q, K, V into a single QKV projection.
  This enables a single fused GEMM instead of 3 separate ones.
  Same for gate_proj + up_proj in the MLP.
  
  Result: Fewer kernel launches, better GPU utilization.
'''
print(weight_mapping)

In [ ]:
# load_weights() implementation for LLaMA

load_weights_code = '''
# Source: vllm/model_executor/models/llama.py

class LlamaForCausalLM(nn.Module):
    
    def load_weights(self, weights: Iterable[Tuple[str, torch.Tensor]]):
        """Load weights from an iterator of (name, tensor) pairs."""
        
        # Define weight mapping and merging rules
        stacked_params_mapping = [
            # (vllm_name_part, hf_name_part, shard_id)
            (".qkv_proj", ".q_proj", "q"),
            (".qkv_proj", ".k_proj", "k"),
            (".qkv_proj", ".v_proj", "v"),
            (".gate_up_proj", ".gate_proj", 0),
            (".gate_up_proj", ".up_proj", 1),
        ]
        
        params_dict = dict(self.named_parameters())
        
        for name, loaded_weight in weights:
            # Check if this weight needs to be merged
            for (vllm_name, hf_name, shard_id) in stacked_params_mapping:
                if hf_name in name:
                    # Replace HF name with vLLM name
                    name = name.replace(hf_name, vllm_name)
                    
                    # Load into the merged parameter
                    param = params_dict[name]
                    weight_loader = param.weight_loader
                    weight_loader(
                        param, loaded_weight, shard_id
                    )
                    break
            else:
                # Direct 1:1 mapping
                param = params_dict[name]
                weight_loader = getattr(
                    param, "weight_loader",
                    default_weight_loader,
                )
                weight_loader(param, loaded_weight)
'''
print(load_weights_code)

## 7. Memory-Efficient Loading

In [ ]:
efficient_loading = """
Memory-Efficient Loading Techniques
════════════════════════════════════

1. META DEVICE INITIALIZATION:
   - Create model on "meta" device (no memory allocated)
   - Tensors have shapes but no data
   - Then load weights directly into final device
   - Avoids: allocating full model twice (CPU then GPU)

2. STREAMING WEIGHT LOADING:
   - Load weights one tensor at a time from disk
   - Immediately move to GPU and delete CPU copy
   - Peak CPU memory: size of largest single tensor
   - Instead of: entire model in CPU RAM at once

3. MEMORY-MAPPED FILES:
   - Safetensors uses mmap for zero-copy reading
   - OS manages page-in/page-out automatically
   - Only pages actively being read are in RAM

4. TP-AWARE LOADING:
   - For tensor parallelism, only load the shard needed by each GPU
   - Instead of loading full weight and then slicing
   - Reduces peak memory by TP factor

5. DTYPE CONVERSION ON LOAD:
   - Convert to target dtype (e.g., FP16) during loading
   - Avoids having FP32 weights in memory temporarily

Loading Timeline (LLaMA-70B, 4x A100):
  Without optimizations: ~5 minutes, ~300GB peak RAM
  With optimizations:    ~2 minutes, ~40GB peak RAM per GPU
"""
print(efficient_loading)

## 8. Adding a New Model to vLLM

In [ ]:
new_model_guide = """
Adding a New Model Architecture to vLLM
════════════════════════════════════════

Step 1: Create model file
  vllm/model_executor/models/my_model.py

Step 2: Implement the model class
  class MyModelForCausalLM(nn.Module):
      def __init__(self, config, cache_config, quant_config, ...):
          # Use vLLM's TP-aware layers:
          self.qkv = QKVParallelLinear(...)  # Column-parallel
          self.o_proj = RowParallelLinear(...)  # Row-parallel
          self.attn = Attention(...)  # Uses PagedAttention
      
      def forward(self, input_ids, positions, kv_caches, attn_metadata):
          # Standard transformer forward pass
          # Use self.attn for attention (handles PagedAttention)
          ...
      
      def load_weights(self, weights):
          # Map HF weight names to vLLM parameter names
          # Handle merged QKV, TP sharding, etc.
          ...

Step 3: Register in registry.py
  _MODELS["MyModelForCausalLM"] = ("my_model", "MyModelForCausalLM")

Step 4: Test
  python -c "from vllm import LLM; llm = LLM('path/to/my_model')"

Key Requirements:
  - Use vLLM's Attention class (not custom attention)
  - Use TP-aware linear layers (Column/RowParallelLinear)
  - Implement load_weights() with proper weight mapping
  - Support the forward() signature with kv_caches and attn_metadata
"""
print(new_model_guide)

## Exercises

### Exercise 1: Calculate Sharded Weight Sizes

In [ ]:
# Exercise 1: Calculate weight sizes for different TP configurations

def calculate_sharded_sizes(hidden_size, num_heads, num_kv_heads, head_dim,
                            intermediate_size, num_layers, vocab_size,
                            tp_size, dtype_bytes=2):
    """Calculate per-GPU weight sizes with tensor parallelism."""
    
    heads_per_gpu = num_heads // tp_size
    kv_heads_per_gpu = num_kv_heads // tp_size
    intermediate_per_gpu = intermediate_size // tp_size
    
    per_layer = {
        'Q proj': hidden_size * heads_per_gpu * head_dim,
        'K proj': hidden_size * kv_heads_per_gpu * head_dim,
        'V proj': hidden_size * kv_heads_per_gpu * head_dim,
        'O proj': heads_per_gpu * head_dim * hidden_size,
        'Gate proj': hidden_size * intermediate_per_gpu,
        'Up proj': hidden_size * intermediate_per_gpu,
        'Down proj': intermediate_per_gpu * hidden_size,
        'Input LN': hidden_size,
        'Post-attn LN': hidden_size,
    }
    
    non_layer = {
        'Embedding': vocab_size * hidden_size,  # Not sharded (replicated)
        'Final LN': hidden_size,
        'LM Head': vocab_size * hidden_size,  # May be tied with embedding
    }
    
    layer_params = sum(per_layer.values())
    total_params = layer_params * num_layers + sum(non_layer.values())
    total_bytes = total_params * dtype_bytes
    
    return {
        'per_layer': {k: v * dtype_bytes / 1024**2 for k, v in per_layer.items()},
        'non_layer': {k: v * dtype_bytes / 1024**2 for k, v in non_layer.items()},
        'total_per_layer_mb': layer_params * dtype_bytes / 1024**2,
        'total_gb': total_bytes / 1024**3,
    }

# LLaMA-70B
config = {
    'hidden_size': 8192, 'num_heads': 64, 'num_kv_heads': 8,
    'head_dim': 128, 'intermediate_size': 28672,
    'num_layers': 80, 'vocab_size': 32000,
}

for tp in [1, 2, 4, 8]:
    result = calculate_sharded_sizes(**config, tp_size=tp)
    print(f"\nTP={tp}: {result['total_gb']:.1f} GB per GPU")
    if tp == 4:
        print("  Per-layer breakdown (MB):")
        for name, size_mb in result['per_layer'].items():
            print(f"    {name}: {size_mb:.1f} MB")

### Exercise 2: Weight Mapping Challenge

In [ ]:
# Exercise 2: Implement weight name mapping

def map_hf_to_vllm(hf_name: str) -> tuple:
    """Map a HuggingFace weight name to vLLM name and shard info."""
    
    mappings = [
        ('.q_proj.', '.qkv_proj.', 'q'),
        ('.k_proj.', '.qkv_proj.', 'k'),
        ('.v_proj.', '.qkv_proj.', 'v'),
        ('.gate_proj.', '.gate_up_proj.', 0),
        ('.up_proj.', '.gate_up_proj.', 1),
    ]
    
    for hf_part, vllm_part, shard_id in mappings:
        if hf_part in hf_name:
            return hf_name.replace(hf_part, vllm_part), shard_id
    
    return hf_name, None  # Direct mapping

# Test
test_names = [
    'model.layers.0.self_attn.q_proj.weight',
    'model.layers.0.self_attn.k_proj.weight',
    'model.layers.0.self_attn.v_proj.weight',
    'model.layers.0.self_attn.o_proj.weight',
    'model.layers.0.mlp.gate_proj.weight',
    'model.layers.0.mlp.up_proj.weight',
    'model.layers.0.mlp.down_proj.weight',
    'model.embed_tokens.weight',
]

print(f"{'HuggingFace Name':<50} {'vLLM Name':<50} {'Shard':<6}")
print("=" * 106)
for name in test_names:
    vllm_name, shard = map_hf_to_vllm(name)
    print(f"{name:<50} {vllm_name:<50} {str(shard):<6}")

## Summary

Key takeaways:

1. **Model loading pipeline**: Registry lookup → create empty model → iterate weights → shard → load to GPU
2. **Safetensors**: Preferred format for security and performance (mmap, lazy loading)
3. **Weight sharding**: Column-parallel (QKV) and row-parallel (output) for tensor parallelism
4. **Quantized loading**: GPTQ/AWQ weights need repacking for CUDA kernel compatibility
5. **Weight merging**: Q/K/V merged into QKV; gate/up merged into gate_up for fewer kernels
6. **Memory efficiency**: Meta device, streaming loading, and mmap minimize peak memory

**Next**: Chapter 3.8 explores Attention Backends.